In [2]:
import sys,os
import numpy as np
from crispy.tools.initLogger import getLogger
log = getLogger('crispy')

This notebook is intended to be a sandbox that demonstrates functionality illustrated at the following webpage:
https://mjrfringes.github.io/crispy/notebooks/Introduction.html

## 1. Initialization

In [ ]:
# Set our directory to the crispy install location
os.chdir("C:/Users/ebray/Github_Repos/crispy/crispy/")
from crispy.configs.WFIRST.params import Params

# Initialize our Params object and print locations of a few directories
par = Params()
print(f'wavecalDir is {par.wavecalDir}')
print(f'exportDir is {par.exportDir}')
print(f'unitTestOutputs is {par.unitTestsOutputs}')

wavecalDir is ..//ReferenceFiles/wavecalR50_770/
exportDir is ..//SimResults
unitTestOutputs is ..//unitTestsOutputs


## 2. Create the flatfield

In [ ]:
from crispy.unitTests import testCreateFlatfield
help(testCreateFlatfield)

#### 2.1 Determine wavelengths at which to make the flatfield

In [ ]:
from crispy.tools.reduction import calculateWaveList
#help(calculateWaveList)
lam_midpts,lam_endpts = calculateWaveList(par)
print(lam_midpts)

#### 2.2 Actually create the flatfield

In [ ]:
testCreateFlatfield(par,useQE=False)

In [ ]:
# And also display the newly-appended-to header parameters
par.hdr

In [ ]:
par.lenslet_sampling

#### 2.3 Display some results

##### 2.3.1 Starting with the input flatfield cube passed through the IFS

In [ ]:
from crispy.tools.image import Image
import matplotlib.pyplot as plt
plt.close('all')
%matplotlib qt
fig,ax = plt.subplots(figsize=(8,7))
image_filepath = par.unitTestsOutputs+'/flatfield.fits'
img = Image(image_filepath).data
im = ax.imshow(img,cmap='gray')
cbar = plt.colorbar(im)
fig.tight_layout()
plt.show()

# Also display a zoomed-in portion in higher detail. 
plt.figure(figsize=(6,6))
subsize = 50
plt.imshow(img[par.npix//2-subsize:par.npix//2+subsize,par.npix//2-subsize:par.npix//2+subsize],cmap='gray')
plt.colorbar()
plt.show()

### 3 Simulate Detector Readout

In [ ]:
from crispy.tools.detector import readDetector,averageDetectorReadout
par.nonoise=False  # turn off photon counting otherwise things are strange
par.EMStats =False # turn off EM register statistics
par.PCmode = False # turn off photon counting threshold
read=readDetector(par,Image(image_filepath),inttime=100)


plt.figure(figsize=(6,6))
subsize = 15

plt.imshow(read[par.npix//2-subsize:par.npix//2+subsize,par.npix//2-subsize:par.npix//2+subsize],cmap='gray')
plt.colorbar()

In [ ]:
# Let’s save the noisified frame to a new name.
newImage = Image(data=read,header=par.hdr)
newImage.write(par.unitTestsOutputs+'/flatfield_noise.fits',overwrite=True)

### 4 Spectral Extraction

In [ ]:
# The reduction step is straightforward, as long as the wavelength calibration is good.
from crispy.IFS import reduceIFSMap
cube = reduceIFSMap(par,par.unitTestsOutputs+'/flatfield_noise.fits')

In [ ]:
# Now we can display the cube interactively, or look it up with DS9 (it is located in par.exportDir)

import ipywidgets
def plt_ifs_optext(wchan):
    plt.clf()  # Clear the current figure
    im = plt.imshow(cube.data[wchan-1,:,:], cmap='gist_heat')
    plt.colorbar(im)  # Add a new colorbar for the current image
ipywidgets.interact(plt_ifs_optext, wchan=(1,cube.data.shape[0]));